<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/solutions/notebook.ipynb)


# Session 2 — Call a model through the adapter

**Goal:** call a model through one method, run the same prompt on two lanes, and turn a provider failure into a refusal instead of a traceback. *Thread: harness engineering.*

Everything runs on the deterministic `FakeLLM` by default. `LIVE` is whichever lane your `.env` names; when that lane is not reachable the preflight says so and `LIVE` is a `FakeLLM` too. No cell here needs a key, a network, or a paid call.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. One call through the seam

A system instruction, a user question, a text answer. `complete(system, user) -> str` is the whole boundary, and every framework wraps it. `FakeLLM` answers from a keyword table, so the same question always gets the same answer.

In [ ]:
from bootcamp_agent.llm import FakeLLM

hello_llm = FakeLLM(
    responses={
        "hello": "Hello! I am a deterministic stand-in for a language model.",
        "agent": "An agent is a loop around a model: perceive, decide, act, observe.",
    },
    default="I have no canned answer for that — a real model would improvise here.",
)

print(hello_llm.complete(system="You are concise.", user="Say hello to the bootcamp"))

## 2. Exercise: three questions, three paths

**Context.** `hello_llm` knows two keywords, `agent` and `hello`, matched case-insensitively in insertion order. Anything else gets the default. Three calls show all three paths.

**Instructions.**

1. Case 1 is done: a question containing `agent`.
2. Add case 2: a question containing `hello`.
3. Add case 3: a question that matches neither keyword.
4. Run the cell, read the three answers, then run the check.

In [ ]:
answers = []
answers.append(hello_llm.complete(system="You are concise.", user="What is an agent?"))  # case 1, done
answers.append(hello_llm.complete(system="You are concise.", user="hello there"))
answers.append(hello_llm.complete(system="You are concise.", user="What is the moon made of?"))

for reply in answers:
    print(reply)

**Expected output** (yours may differ in wording, not in shape):

```
An agent is a loop around a model: perceive, decide, act, observe.
Hello! I am a deterministic stand-in for a language model.
I have no canned answer for that — a real model would improvise here.
✅ ch02-e1 passed
```

In [ ]:
check("ch02-e1", answers)

## 3. The seam itself

`FakeLLM`, `OllamaClient`, `AnthropicClient` and `OpenAICompatibleClient` satisfy the same one-method protocol. Nothing inherits from it: a class is an `LLMClient` because it has `complete(system, user) -> str`. Swapping providers changes no application code. `LIVE`, from the preflight cell, is whichever lane this machine could actually reach.

In [ ]:
import inspect

from bootcamp_agent import llm

print(inspect.getsource(llm.LLMClient))
print(f"LIVE is a {type(LIVE).__name__}")

## 4. Exercise: the same prompt, two lanes

**Context.** The fake is deterministic: same question, same answer, every time. A real model is not. That gap is why week 2 spends a session on evaluation — you measure properties, not words.

This cell needs no provider. Without one, `LIVE` is a `FakeLLM` and the live pair comes back identical; that is a correct run, not a skipped one.

Without a provider, `LIVE` is a bare `FakeLLM` and its default answer is the course's refusal JSON. Session 3 is where that string stops being text and becomes a parsed contract.

**Instructions.**

1. The `fake` pair is done: the same question twice through `hello_llm`.
2. Fill the `live` pair: the same question twice through `LIVE`.
3. Run the cell. Compare the two pairs. Then run the check.
4. On the ollama lane the two live answers usually differ in wording. On the fake lane they are identical. Both are correct.

In [ ]:
question = "Explain in one sentence what an agent is."

runs = {
    "fake": [hello_llm.complete(system="You are concise.", user=question) for _ in range(2)],  # done
    "live": [LIVE.complete(system="You are concise.", user=question) for _ in range(2)],
}

for lane, pair in runs.items():
    print(f"[{lane}] same answer twice? {pair[0] == pair[1] if len(pair) == 2 else 'not run yet'}")
    for reply in pair:
        print(f"   {reply[:100]}")

**Expected output** (yours may differ in wording, not in shape):

```
[fake] same answer twice? True
   An agent is a loop around a model: perceive, decide, act, observe.
   An agent is a loop around a model: perceive, decide, act, observe.
[live] same answer twice? False        <- True on the fake lane, usually False on ollama
   An agent is a program that uses a model to decide which actions to take toward a goal.
   An agent is software that plans, calls tools, and checks results until a task is done.
✅ ch02-e2 passed
```

In [ ]:
check("ch02-e2", runs)

## 5. Exercise: your reliability bar

**Context.** Reliability is a decision you make before the first real answer, not after. Write yours down so the evaluation week has a target.

**Instructions.**

1. The first sentence is done as an example. Replace it with your own.
2. Write the other two sentences. Be concrete: name a kind of task, a kind of data, a kind of risk.
3. Run the check. It only measures that each sentence is yours and complete.

In [ ]:
reliability_bar = {
    "reliable_when": "it cites a corpus document I can open, or plainly says it does not know.",
    "review_when": "the answer would change money, credentials, or anything in production.",
    "never_unreviewed": "rotate a secret or delete a branch on the shared repository.",
}
for key, sentence in reliability_bar.items():
    print(f"{key:18} {sentence}")

**Expected output** (yours may differ in wording, not in shape):

```
reliable_when      it cites a corpus document I can open, or plainly says it does not know.
review_when        the answer would change money, credentials, or anything in production.
never_unreviewed   rotate a secret or delete a branch on the shared repository.
✅ ch02-e3 passed
```

In [ ]:
check("ch02-e3", reliability_bar)

## 6. Failure injection: the two configuration failures

**Not scored, and run it anyway.** Both failures happen inside your process, before any request leaves the machine, so they cost nothing and are the same on every lane.

Watch two things in the output: the missing-key message names `.env.example` and never the key, and the unknown-provider message lists the lanes that do exist. An error that carries its own fix is the standard for the rest of the course.

In [ ]:
from bootcamp_agent.config import ConfigError, Settings, load_settings
from bootcamp_agent.llm import get_client

# 1. A lane that needs a key, with no key in the environment.
try:
    get_client(Settings(provider="anthropic", model=None, api_key=None, base_url=None))
except ConfigError as error:
    print("missing credential ->", error)

# 2. A provider name this course does not serve.
try:
    load_settings(env={"BOOTCAMP_PROVIDER": "gpt-9"})
except ConfigError as error:
    print("unsupported lane   ->", error)

An unsupported **model** is the third case, and it cannot be answered in your process: only the provider knows what it serves. The local lane asks it, without sending a prompt.

The next cell guards itself. Without `BOOTCAMP_PROVIDER=ollama` it prints a skip line and moves on.

In [ ]:
import os

if os.environ.get("BOOTCAMP_PROVIDER", "fake") != "ollama":
    print("skipped: no local lane configured (BOOTCAMP_PROVIDER is not 'ollama')")
else:
    from bootcamp_agent.ollama import DEFAULT_MODEL, probe

    result = probe(model=DEFAULT_MODEL)
    print(f"reachable={result.reachable} model_present={result.model_present}")
    print(result.fix or f"{DEFAULT_MODEL} is pulled and ready")

## 7. Exercise: a deadline that becomes a refusal

**Context.** A provider that never answers is the failure every production agent meets. `urllib` raises `TimeoutError` when the deadline passes; `OllamaClient` catches that and raises `OllamaError`, which is why you catch both. The caller should get one shape either way, so the timeout leaves as a value, not an exception.

**Instructions.**

1. `TimeoutLLM` and the imports are written for you. It stands in for any provider that runs past its deadline.
2. Write `answer_with_timeout(question)`: call `client.complete(...)`, catch `TimeoutError` and `OllamaError`, and return an `AgentResult` whose answer says the model did not respond in time, cites nothing, has confidence `0.0`, and sets `needs_human_review=True`.
3. Keep the cause in the trace and out of the answer. The adapter cannot tell "too slow" from "no server", so do not claim which one it was.
4. Run the cell: it must print, not raise. Then run the check. It calls your function and refuses an escaped exception, a return value that is not an `AgentResult`, a missing review flag, or any citation.

In [ ]:
from bootcamp_agent.agent import AgentResult, TraceEvent
from bootcamp_agent.ollama import OllamaError
from bootcamp_agent.schema import ResearchAnswer


class TimeoutLLM:
    """A provider that never answers: the deadline passes and the transport raises."""

    def complete(self, system: str, user: str) -> str:
        raise TimeoutError("no response within the deadline")


def answer_with_timeout(question: str) -> AgentResult:
    client = TimeoutLLM()
    try:
        text = client.complete(system="You are concise.", user=question)
    except (TimeoutError, OllamaError) as error:
        return AgentResult(
            answer=ResearchAnswer(
                answer="The model did not respond in time.",
                citations=(),
                confidence=0.0,
                needs_human_review=True,
            ),
            trace=(TraceEvent("decision", f"provider failed: {error}"),),
        )
    return AgentResult(
        answer=ResearchAnswer(
            answer=text, citations=(), confidence=0.5, needs_human_review=False
        ),
        trace=(TraceEvent("llm_call", "answered inside the deadline"),),
    )


try:
    result = answer_with_timeout("How does chunking work in RAG?")
    print("DEFENDED —", result.answer.answer)
except NotImplementedError:
    print("not written yet")

**Expected output** (yours may differ in wording, not in shape):

```
DEFENDED — The model did not respond in time.
✅ ch02-e4 passed
```

In [ ]:
check("ch02-e4", answer_with_timeout)

## 8. Bonus: a refusal you can actually debug

**Optional, and above full marks.** Section 7 takes you to 400 of 400. This one
is worth nothing and is excluded from every total — `ch02` is four exercises
whether you do it or not.

**The problem with the refusal you just wrote.** It is correct, and a call that
died instantly and one that died after thirty seconds produce **the same two
lines**. Nobody can act on that.

**What this asks for.** Two facts, in the `trace` and never in the answer:

1. **Which lane failed** — name the provider in a `TraceEvent` detail
2. **How long it waited** — measure it with `time.monotonic()`

**And the trap it will refuse.** Do not write "the model timed out" in the
*answer*. The adapter cannot tell "too slow" from "no server at all", and a
refusal that guesses a cause is a refusal nobody can debug. Say less than you
know, in the place where the reader is looking for it.

Edit `answer_with_timeout` above, then run the cell below.

In [ ]:
from bootcamp_agent.bonus import bonus

# Ships FAILING, on purpose. There is nothing to earn in a cell that arrives
# green — and the message below names exactly what is missing.
bonus("ch02", answer_with_timeout)

## Exit ticket

- What works? What is unclear? What is your next action?
- Homework: run the same prompt on a second lane, and write one sentence on what changed and one on what did not. Then pick the deadline you would give a chat answer, and say what your code does when it passes.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch02")